# QCC v58_c40b — reproduction of LB 23.73311

**All credit for the circuits goes to [heliosli](https://www.kaggle.com/heliosli).**
This notebook is an independent, fully credited reproduction of heliosli's public
portfolio **v58_c40b**, released in the dataset
[heliosli/qcc-circuits-live](https://www.kaggle.com/datasets/heliosli/qcc-circuits-live)
and assembled by the author's kernel
[the-dataset-is-public-now-enjoy](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy).
The per-circuit upstream lineage (Ehan Mallick, ahoirg / SomeJoint, Lera / marculera,
yoyobar, Alex Chilton, …) is listed in the Credits section at the end, verbatim
from the release dataset. All components are Apache-2.0.

## What this notebook does

1. Reads the release manifest `QCC_RELEASE_MANIFEST.json` from the attached dataset.
2. Materialises the 18 portfolio files **in manifest order** (decompressing
   `dqi.kmb.xz` with lzma), verifying every file against the manifest's pinned
   SHA-256 — the same assembly procedure as the author's kernel.
3. Compares every assembled file against the pinned SHA-256 table of our
   submission #8 (below), so you can see exactly which bytes scored 23.73311.
4. Writes `/kaggle/working/submission.zip` (ZIP_DEFLATED, manifest order).

Full official-verifier verification is intentionally **not** re-run inside this
kernel (the dqi full run is 102,400 shots and needs ~6 GB per worker). The
guarantee here is byte identity: every file is accepted only if its SHA-256
matches the pin of a bundle whose score is proven on the leaderboard. Our local
verification with the unmodified official verifiers is summarised below.

## Score achieved with these bytes

| | |
|---|---|
| Our file submission | ref **57857480**, submitted 2026-11-01 11:10 JST |
| Public LB score | **23.73311** (115th place, inside the bronze zone) |
| Author's own submission of v58_c40b | 23.73311 (identical score, large tie cluster) |
| Local estimate (geometric mean of ratios) | ≈ 23.733 (calibration vs LB: +0.00011) |

Author-published ratios for v58_c40b: invsqrt 20.650 · dqi 63.141 · dfthcsos 10.252.

## ⚠️ Dataset version note (please read)

Our submission #8 was built from dataset **v39**. On 2026-11-01 05:38 UTC the
author refreshed the dfthcsos table weights **in place** (dataset **v40**, same
portfolio name `v58_c40b`): 17 of 18 files — every circuit — are byte-identical
to v39, but `dfthcsos/submission.npz` changed
(SHA-256 `89357442…` → `1c48da82…`; arrays `W`, `H2_I`, `offset_W` updated).
We verified the new weights locally with the full official verifier: **PASS**,
cost 327,053,613.24 (λ_eff 47.0619) vs 327,300,810.34 for the #8 weights — i.e.
assembling from the current dataset version yields a slightly *improved*
variant (expected LB ≈ 23.739 by the same geometric-mean estimate), not the
exact #8 bytes. The comparison cell below reports the match status of every
file against the #8 pins, so the output of any run is unambiguous.


In [ ]:
import glob, hashlib, json, lzma, os, shutil, zipfile, pathlib

def find_release():
    hits = glob.glob('/kaggle/input/**/QCC_RELEASE_MANIFEST.json', recursive=True)
    if not hits:
        raise SystemExit('release dataset not found (attach heliosli/qcc-circuits-live)')
    return os.path.dirname(hits[0])

DS = find_release()
WORK = pathlib.Path('/kaggle/working')
SUB = WORK / 'submission'
MANIFEST = json.load(open(os.path.join(DS, 'QCC_RELEASE_MANIFEST.json')))
EXPECTED_PORTFOLIO = 'v58_c40b'
assert MANIFEST['portfolio'] == EXPECTED_PORTFOLIO, (MANIFEST['portfolio'], EXPECTED_PORTFOLIO)
print('release dataset:', DS, '| portfolio:', MANIFEST['portfolio'], '| salt:', MANIFEST['salt'])


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()

if SUB.exists():
    shutil.rmtree(SUB)
assembled = {}
for item in MANIFEST['files']:
    src, dst = os.path.join(DS, item['stored_as']), SUB / item['file']
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if item['stored_as'].endswith('.xz'):
        with lzma.open(src) as fi, open(dst, 'wb') as fo:
            shutil.copyfileobj(fi, fo, 1 << 22)
    else:
        shutil.copyfile(src, dst)
    got = sha256(dst)
    assert got == item['sha256'], ('manifest pin mismatch', item['file'], got)
    assert os.path.getsize(dst) == item['bytes'], ('size mismatch', item['file'])
    assembled[item['file']] = got
    print('sha256 OK (manifest pin)', item['file'], got)
print(len(assembled), '/ 18 files materialised and matched to the manifest by SHA-256')


In [ ]:
# SHA-256 pins of submission #8 (dataset v39), which scored Public 23.73311 (ref 57857480).
SUBMISSION_8_PINS = {
    "dfthcsos/angle_k_0.kmb": "d52f249c5ddedc698c3add5a6cfdf7bb2a66b50e157927a1c17d06988a95e259",
    "dfthcsos/angle_k_1.kmb": "0d16c5108e48e40931d87f1e3ee12d4486e932b341989ce909a47b032e5a7745",
    "dfthcsos/angle_k_10.kmb": "e2668bf5666ca1e49541bef7a255b2100a8a513188fadef3a8dd4e892cfe2094",
    "dfthcsos/angle_k_11.kmb": "5f69f385cb2040e7124e11e1cedb7cea0e0ffa6ad13abe81b2a337744cb9c2ac",
    "dfthcsos/angle_k_12.kmb": "f86995a0ea1990a11a43f1812f47e00a77658f71cc544d44bd5003e616111272",
    "dfthcsos/angle_k_13.kmb": "4390c8114e191b0c74ccd8b6302e5e0ded4e8cb412a3f146eea9887551658876",
    "dfthcsos/angle_k_14.kmb": "62bec7cf502a9e5d49823f8f571164dbd8a56c1d5fc0e46a7f93ff200d57b156",
    "dfthcsos/angle_k_2.kmb": "b7b73491d3aeb59c70d023704df9f2ad69eb96c9d1f9e9920d33d60bd7852d11",
    "dfthcsos/angle_k_3.kmb": "c2437ad406d3772e961e7229580fba7b68c036b97ce36b4067c865e3af729f00",
    "dfthcsos/angle_k_4.kmb": "0df35ae1cb26d3bc09f022b2a397a8b07e5c322283d2c8cf0a53a4bb11b9dd47",
    "dfthcsos/angle_k_5.kmb": "a9a649997bf0696a659c8cbf9e778f92bee2816b6212febc898ed51450ae42c3",
    "dfthcsos/angle_k_6.kmb": "715673382c491b796865dbba235687eafebd16c745872d20f7ae1808f88a62ae",
    "dfthcsos/angle_k_7.kmb": "31e004f932d42750e1dda98abdf3e0097941cb1b1bbd353ed692e52098119cfb",
    "dfthcsos/angle_k_8.kmb": "046ac97b8c6ac9e1895a1da5dc89ea5df55bf487b1822d2ba11697c948145ff9",
    "dfthcsos/angle_k_9.kmb": "10c37a55ab902357cf60c5cf529de0eec9e9497a03f0b2ad7a9888d26d9c1e63",
    "dfthcsos/submission.npz": "893574424ea1b2d8ec2d90e3bd8e922dcbafde3aa4d569d70de129f75106ae60",
    "dqi.kmb": "fb2da4f299c4d41e421e5e57d07915599617291ab1c41d4e0fcefa6db1f2079c",
    "invsqrt.kmb": "bce2948ff24e20fe59666d131f7fc4316a905103937272c52fdd39d494b6fe47"
}

match = 0
for name in MANIFEST['files']:
    f = name['file']
    same = assembled[f] == SUBMISSION_8_PINS[f]
    match += same
    print(('IDENTICAL to submission #8 ' if same else 'DIFFERS from submission #8   '), f)
print(f'{match} / 18 files are byte-identical to submission #8 (LB 23.73311).')
print('Note: since dataset v40 the author refreshed dfthcsos/submission.npz in place;')
print('a single mismatch on that file is expected with the current dataset version and')
print('corresponds to the slightly improved weights described in the header (expected LB ~23.739).')


In [ ]:
zip_path = WORK / 'submission.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for item in MANIFEST['files']:
        z.write(SUB / item['file'], item['file'])
print('wrote', zip_path, zip_path.stat().st_size, 'bytes; zip sha256:', sha256(zip_path))
print('reference: submission #8 zip was 31,379,281 bytes, sha256 33f9d957fc6d44296c475f7f4c6e4e34f9a12f05ae2b088b12859957bd69ff1e')
print('(container hashes depend on zip entry timestamps; per-file pins above are the identity check)')
shutil.rmtree(SUB)


## Local verification record (official verifiers, unmodified)

Run by us locally (Python 3.13 + the competition's kickmix) on the exact
dataset v39 bytes of submission #8:

- **dfthcsos: PASS** at full default settings (87,232 shots, 15 circuits,
  salt `notebook-verification`). Cost 327,300,810.34 (λ_eff 47.097464 ×
  Toffoli 16,237 × Q 428). Λ = 285.25 > 49.0, E_gap = 3.914952 > 3.914898,
  error_1 = 0.011815, error_2 = 0.0199999998 (< 20 mHa). Ratio 10.2523.
- **invsqrt: PASS** on the reduced domain x ∈ [1, 4096] × 2 shots (8,192 shots,
  all passing; Toffoli count is domain-independent). Q = 62, mean Toffoli
  1,325.48, reduced-domain cost 100,567.54. Ratio 20.6505. Worst error ratio
  0.99932 at x = 161. circuit_sha256 = manifest pin.
- **dqi: static checks PASS + reduced simulation PASS.** kickmix loads
  (893 qubits, 63,176,209 instructions, Toffoli-like 482,802), no Z_POW gates,
  official `validate_registers` passes. 2,560 shots at the official
  log2_delta = −128: 0 value/phase errors, ε₊ = 0.03406 < 1/16, mean Toffoli
  407,445.21. circuit_sha256 = manifest pin. The full 102,400-shot run is left
  to official scoring (the identical bytes are LB-proven by the author and a
  large third-party cluster at 23.73311).

Dataset v40 dfthcsos weights (the in-place refresh): full official verify
**PASS**, cost 327,053,613.24, λ_eff 47.0619, error_1 = 0.011824,
error_2 = 0.0199999998, E_gap = 3.914938.


# Credits

# Sources and credits

- **invsqrt** — Ehan Mallick's 62-qubit file, byte for byte, from the output of Alex Chilton's portfolio
  https://www.kaggle.com/code/alexchilton/22-57-lb-verified-three-task-portfolio (Apache-2.0), which credits it to
  https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225 and builds it on yoyobar's
  62-qubit circuit (https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits, Apache-2.0; dataset
  https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031). Earlier lineage, as credited in our previous
  versions: yoyobar's own 62-qubit work on Akmal Xodarev's (busyaprime) and mrmeows / team selenium's circuits, with
  Ahmed Berat Özer's patch as distributed in manwithacat's Quantum Voodoo; earlier: Phuc Thai (DePie), Dmitrii
  Gluzdov, marsyas.

- **dqi** — ahoirg's (Ali Haydar Özdağ) 893-qubit build of SomeJoint's q18 decoder, byte for byte, from the 1 November
  output of https://www.kaggle.com/code/ahoirg/score-23-64254-reproducible-circuit-portfolio (built there from the
  source in SomeJoint's Apache-2.0 dataset https://www.kaggle.com/datasets/somejoint/qcc-dqi-900-undo, with yoyobar's
  one-carry pole-history idea from https://www.kaggle.com/datasets/yoyobar/qcc-local-sota-20261031; upstream notices
  retained there). Lineage as credited there: Agniv Banerjee (SomeJoint) and their team, yoyobar, Ahmed Berat Özer,
  Alex Chilton, jday96314 (QuantumAI), mrmeows, manwithacat, Dmitrii Gluzdov, ViberTeam, Phuc Thai (DePie).
- **dfthcsos** — Lera's (marculera) second (9, 143, 40) factorization: its angle words and its fifteen angle circuits,
  byte for byte, from https://www.kaggle.com/code/marculera/df-thc-10x-compact-groups-from-the-hamiltonian?scriptVersionId=364385620
  (Apache-2.0). Its sources, as credited there: Ahmed Berat Özer (ahmedberatozer), somejoint, mrmeows, heliosli,
  lightningv08, kozykappa, Kozykorpesh and Sumit Chongder, Artem1410 (anazemcev), haideptry via phucthaiv02, Michael
  Rustad, k256net and Dmitrii Gluzdov, and the hosts. The table weights (W, H2_I) are ours, starting from Lera's.
  With thanks to Atakan Aldemir, whose
  https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler pointed us to Lera's
  tables.
- **Organizers** — task definitions, verifiers and kickmix (Google Quantum AI / Qualtran, competition data).

---

*Reproduction published by rdaccel with full credit to heliosli and all upstream
authors above. If any pin in this notebook ever disagrees with the dataset
manifest, trust the manifest of the dataset version you actually downloaded —
and note that only the dataset v39 pins listed in the comparison cell are the
bytes of submission #8 (Public 23.73311, ref 57857480).*
